# Data preparation with Databao – Web shop orders demo (Case 1)

Welcome! This notebook will walk you through a practical end‑to‑end data preparation workflow for a webshop orders dataset using [Databao](https://databao.app).
You’ll learn how to transform transactional into clean, structured, and insight-ready outputs.

The notebook contains the typical data-preparation steps:
> Understanding → Cleaning → Integration → Feature engineering → Aggregation & export → Insights

This corresponds to the following progression:
> “What data do we have?” → “Is it clean?” → “Can we aggregate and group it?” → “What KPIs can we compute?” → “What are the trends and drivers?” → “What actions do we take?”

The notebook contains a DuckDB file with a sample dataset, and it can be used with both local and cloud LLMs. You can learn more about connecting to data, using LLMs, and running Databao in the [Databao docs](https://jetbrains.github.io/databao-docs/).

Let’s dive in!

In [ ]:
# Install Databao and other packages (safe to re-run)
!pip install -q duckdb databao matplotlib pandas


In [ ]:
# Import packages
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Connect to the local DuckDB file. `read_only=False` allows registering temp views/DFs.
DB_PATH = "data/web_shop_raw.duckdb"
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"Expected DuckDB at {DB_PATH}. If you don't have it, check the README for setup instructions."
    )
conn = duckdb.connect(DB_PATH, read_only=False)
print(f"Connected to DuckDB database: {DB_PATH}")


In [ ]:
# Import Databao, LLM config and Context
import databao
from databao import LLMConfig
from databao import Context

### 1. Configure your LLM

Databao supports both cloud and local LLMs.
For this demo, it’s easier and faster to use an OpenAI cloud model, but it requires an API key.

If you prefer to use a local model, all your data remains on your machine, but downloading a model may take some time. Depending on the model you use and your machine specs, generating answers may be slower compared to a cloud model.

For easier setup, this notebook uses a cloud LLM by default. If you prefer to use a local LLM, uncomment the corresponding section and comment out the line with the cloud LLM config.


In [ ]:
# Add your OpenAI API key. Comment out the following line if you prefer to use a local model
%env OPENAI_API_KEY=<YOUR_API_KEY>

In [ ]:
# Option A — Cloud model (OpenAI). Low temperature helps produce deterministic SQL/plots
llm_config = LLMConfig(name="gpt-5.1", temperature=0)

# Option B — Local model (Ollama)
# llm_config = LLMConfig.from_yaml("../configs/qwen3-8b-ollama.yaml")  # Use a custom config file

### 2. Create a Databao agent and register sources and context

The following cell registers:
- A DuckDB connection
- A small per‑source schema overview (Markdown file)
- General project‑wide context via `agent.add_context()`


In [ ]:
context_builder = Context.builder()

# Add a DuckDB connection with source context
context_builder.add_db(conn)

# Add additional project‑wide context (not tied to a specific source)
context_builder.add_context(
    """
    Project‑wide notes:
    - Monetary values are in EUR, unless stated otherwise.
    - Orders with a status of 'canceled' should be excluded from KPIs unless they're explicitly requested.
    - Treat obviously fake/test rows as data quality issues (e.g., emails like test@, or id like 'Test User').
    - Date columns use UTC timestamps unless noted; compute delivery_days from purchase to delivered_customer.
    """
)

context = context_builder.build()
agent = databao.agent(context, llm_config=llm_config)

print("Registered DBs:", list(agent.dbs.keys()))


### 3) Start a thread

- `.ask(prompt)` executes immediately (eager mode) and materializes results.
- If you want to chain several `ask()` calls before executing them, you can switch to lazy mode using `session.thread(lazy=True)`.


In [ ]:
thread = agent.thread()

## Step 1 — Understanding data

Question: What data do we have, and how are the tables connected?
Action: Explore schema (orders, items, products, payments, reviews, customers, sellers).
Outcome: Identify key joins (order_id, customer_id, seller_id).


In [ ]:
thread.ask(
    """
    Provide a concise schema overview of our webshop database. Include the following:
    - List of core tables (orders, order_items, products, payments, reviews, customers, sellers).
    - Primary keys and foreign keys per table.
    - A short note on how to join them at the order level.
    Return a short Markdown/text summary, keep it tight.
    """
)


In [ ]:
display(Markdown(thread.text()))

In [ ]:
# Ask for quick row counts to gauge table sizes
thread.ask(
    """
    Return a dataframe of row counts for the main tables
    """
)

In [ ]:
row_counts = thread.df()
row_counts

## Step 2 — Cleaning and validation

Question: Are there duplicates or missing values?
Action: Fix data types, clean nulls, remove test rows.
Outcome: Normalized, ready-to-merge datasets.

We will ask Databao to propose cleaning steps and produce cleaned, queryable outputs (views or temp tables) we can reuse.


In [ ]:
thread.ask(
    """
    Check tables for common issues and propose pragmatic cleaning steps:
    - Duplicates by natural keys (e.g., one row per order_id in orders etc.).
    - Obvious type issues (timestamps, numerics).
    - Clearly fake/test rows (e.g., customers with emails like '%test%' or ids 'test%').
    show me tables with issues and write down what's wrong with them
    -
    """
)

In [ ]:
display(Markdown(thread.text()))

In [ ]:
thread.ask(
    """
    give me sql to fix issues, mentioned above
    """
)

## Step 3 — Integration

Question: Can we combine customers, sellers, payments, and reviews?
Action: Join entities into a consolidated dataset.
Outcome: Unified order-level table with spend, freight, category, seller location, and reviews.


In [ ]:
thread.ask(
    """
    Build a unified order-level dataset from the cleaned tables with one row per order_id. Include at least:
    - order_id, order_purchase_timestamp, order_approved_at
    - customer_id, customer_city, customer_state
    - seller_id (dominant seller for the order if multiple), seller_city, seller_state
    - items_count, total_items_price, total_freight_value, total_payment_value
    - main_product_category_name (mode across items), review_score
    Name this output orders_unified (as a DuckDB temp table) and also return it as a DataFrame sample (head).
    """
)


In [ ]:
orders_unified = thread.df()
orders_unified

## Step 4 — Feature engineering

Question: Which KPIs help analyze performance?
Action: Compute total_price, freight, delivery_days, delay_days, review_score.
Outcome: Metrics such as average delivery time, review score, and order value.


In [ ]:
thread.ask(
    """
    From orders_unified, compute a feature-rich table orders_features with per-order KPI: come up with most important metrics, expain why you chose it
    """
)


In [ ]:
display(Markdown(thread.text()))

In [ ]:
orders_features = thread.df()
orders_features

## Step 5 — Aggregation, grouping, and export

Question: Can we summarize results per category or seller?
Action: Aggregate by category, month, and seller.
Outcome: Analytical dataset ready for KPI dashboards or modeling.


In [ ]:
# Aggregate by month and product category
thread.ask(
    """
    From orders_features join back the main_product_category_name (if not already present) and aggregate by:
    month = date_trunc('month', order_date), product_category_name.
    Compute: orders_count, revenue_total, aov
    Return as df_category_month.
    """
)


In [ ]:
df_category_month = thread.df()
df_category_month.head()

In [ ]:
# Aggregate by seller overall
thread.ask(
    """
    Aggregate performance by seller_id using orders_features
    """
)


In [ ]:
df_seller_kpis = thread.df()
df_seller_kpis.head()


### Wrapping it up

- You just walked through the complete data preparation workflow in Databao: Understanding → Cleaning → Integration → Feature Engineering → Aggregation & Export.
- You used both per‑source and project‑wide contexts to guide the LLM and ensure consistent results.
- You used `.ask()` in the default eager mode to materialize results incrementally. You can also try to chain multiple asks and compute them at once in a new thread with `lazy=True`.
- Your final datasets are now analytics-ready and suitable for dashboards, exploratory analysis, or downstream machine-learning pipelines.
